In [1]:
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_distances, cosine_similarity
from sklearn.metrics import pairwise_distances, silhouette_score
from sklearn.cluster import KMeans
from pathlib import Path
from IPython.display import display

# Install sentence-transformers in the notebook environment if it's missing.
%pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

nltk.download("all")


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


/home/h00pyfr00d/Documents/AML_Spotify/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /home/h00pyfr00d/nltk_data...
[nltk_data]    | 

True

In [2]:
data = pd.read_csv("data/additional_features.csv")
# remove songs with missing lyrics and reset the index
data = data.dropna(subset=['lyrics']).reset_index(drop=True)
data

,id,Rank,Title,Primary Artists,Featuring Artists,Date,Danceability,Energy,Loudness,Speechiness,...,lyrics,detected_languages,num_languages,language_confidences,artist_id,explicit,track_popularity,artist_followers,artist_genres,artist_popularity
0,47EiUVwUp4C9fGccaPuUCS,1,dákiti,"bad bunny, jhayco",NaN,26/11/2020,0.731,0.573,-10059.00,0.054,...,"Baby, ya yo me enteré, se nota cuando me ve'\n...",SPANISH,1,SPANISH: 1.0,4q3ewBCX7sLwd24euuV69X,True,78,102185553.0,"reggaeton, trap latino, urbano latino, latin",95.0
1,7igeByaBM0MgGsgXtNxDJ7,1,positions,ariana grande,NaN,29/10/2020,0.736,0.802,-4759.00,0.086,...,Heaven sent you to me\nI'm just hopin’ I don't...,ENGLISH,1,ENGLISH: 1.0,66CXWjxzNUsdJxJ2JdwvnR,True,0,107261237.0,pop,90.0
2,3qQbCzHBycnDpGskqOWY0E,1,ella baila sola,"eslabon armado, peso pluma",NaN,29/05/2023,0.668,0.758,-5176.00,0.033,...,"Compa, ¿qué le parece esa morra?\nLa que anda ...",SPANISH,1,SPANISH: 1.0,0XeEobZplHxzM9QzFQWLiR,False,73,6434994.0,"corridos tumbados, sad sierreño, corrido, sier...",75.0
3,0Yde1VrWbGF2Tus2AQhMuT,1,circles,post malone,NaN,06/09/2019,0.703,0.756,-3407.00,0.038,...,"Oh, oh, oh\nOh, oh, oh\nOh, oh, oh, oh, oh\n...",ENGLISH,1,ENGLISH: 1.0,246dkjvS1zLTtiykXe5h60,False,0,47686394.0,NaN,85.0
4,0TK2YIli7K1leLovkQiNik,1,señorita,"camila cabello, shawn mendes",NaN,25/07/2019,0.759,0.540,-6039.00,0.029,...,I love it when you call me señorita\nI wish I ...,ENGLISH,1,ENGLISH: 1.0,7n2wHs1TKAczGzO7Dd2rGr,False,79,45665812.0,NaN,81.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8029,7wBJfHzpfI3032CSD7CE2m,200,stargazing,travis scott,NaN,14/05/2020,0.487,0.789,-3.69,0.044,...,"Ah\n\nRollin', rollin', rollin', got me starga...",ENGLISH,1,ENGLISH: 1.0,0Y5tJX1MQlPlqiwlOH1tJY,True,73,41006554.0,rap,89.0
8030,7yotKA30dwTKNEGomV9ZsI,200,kod,j. cole,NaN,28/06/2018,0.823,0.783,-6274.00,0.154,...,"Go, go, go\nGo, go, go\nGo, go, go\nGo, go, go...",ENGLISH,1,ENGLISH: 1.0,6l3HvQ5sa6mXTsMTB19rO5,True,60,27037715.0,rap,82.0
8031,5FOCkGt7bp22nBVfQ4WSIT,200,indecente,anitta,NaN,19/04/2018,0.874,0.602,-5118.00,0.127,...,"Una mirada te delata, un perfume que te atrapa...",SPANISH,1,SPANISH: 1.0,7FNnA9vBm6EKceENgCGRMb,False,48,13245754.0,"brazilian pop, funk pop, brazilian funk, funk ...",78.0
8032,7zQirOExB0VR8yWUOqYeio,200,dolla sign slime (feat. megan thee stallion),lil nas x,megan thee stallion,27/09/2021,0.923,0.533,-6016.00,0.262,...,Day-Day-Daytrip took it to ten (\nHey\n)\n\nY'...,ENGLISH,1,ENGLISH: 1.0,7jVv8c5Fj3E9VhNjxT4snq,True,50,12311640.0,NaN,71.0


In [3]:
# Build semantic embeddings for lyrics, find similar songs, and train a simple prediction model.
# Uses sentence-transformers if available, otherwise falls back to TF-IDF.
lyrics = data['lyrics'].fillna("").astype(str).tolist()

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
embeddings_sparse = vectorizer.fit_transform(lyrics)
embeddings = embeddings_sparse  # keep sparse matrix
embeddings_is_sparse = True

nn = NearestNeighbors(n_neighbors=6, metric='cosine', n_jobs=-1)
nn.fit(embeddings)

def get_similar(song_idx: int, k: int = 5):
    """
    Return k most similar songs (index, distance, title, primary artists).
    Excludes the query itself from the results.
    """
    query = embeddings[song_idx] if embeddings_is_sparse else embeddings[song_idx]
    distances, indices = nn.kneighbors([query] if not embeddings_is_sparse else query, n_neighbors=k+1)
    distances = distances[0]
    indices = indices[0]
    results = []
    for dist, idx in zip(distances, indices):
        if idx == song_idx:
            continue
        row = data.iloc[idx]
        results.append({'idx': int(idx), 'distance': float(dist), 'title': row['Title'], 'artists': row['Primary Artists']})
        if len(results) >= k:
            break 
    return results

In [4]:
def get_songs_by_artist(artist: str) -> pd.DataFrame:
    a = artist.strip().lower()
    prim = data['Primary Artists'].fillna('').astype(str)
    feat = data['Featuring Artists'].fillna('').astype(str)

    pattern = re.escape(a)
    mask_prim = prim.str.lower().str.contains(pattern, na=False)
    mask_feat = feat.str.lower().str.contains(pattern, na=False)

    return data[mask_prim | mask_feat].copy()

def get_lyrics_by_title(title: str, exact: bool = True, first_only: bool = True):
    t = title.strip().lower()
    titles = data['Title'].fillna('').astype(str).str.lower()
    if exact:
        mask = titles == t
    else:
        mask = titles.str.contains(re.escape(t), na=False)
    matches = data[mask]

    if matches.empty:
        return None if first_only else matches
    if first_only:
        row = matches.iloc[0]
        return row['lyrics']
    
    return matches[['lyrics']].copy()

features = ["Danceability", "Energy", "Loudness", "Speechiness", "Acousticness", "Instrumentalness", "Valence"]

In [5]:
# Global variable to cache semantic embeddings
semantic_embeddings_cache = None

def ensure_semantic_embeddings():
    """
    Ensure semantic embeddings are available, either from cache or by computing them.
    Returns dense embeddings for all lyrics in the dataset.
    """
    global semantic_embeddings_cache
    
    if semantic_embeddings_cache is not None:
        return semantic_embeddings_cache
    
    try:
        # Check if we have pre-computed embeddings saved
        embeddings_file = "data/semantic_embeddings.npy"
        if Path(embeddings_file).exists():
            print("Loading pre-computed semantic embeddings...")
            semantic_embeddings_cache = np.load(embeddings_file)
            return semantic_embeddings_cache
        
        # Compute semantic embeddings using sentence transformers
        print("Computing semantic embeddings using sentence-transformers...")
        model = SentenceTransformer('all-MiniLM-L6-v2')
        
        # Get lyrics and ensure they're strings
        lyrics_list = data['lyrics'].fillna("").astype(str).tolist()
        
        # Compute embeddings in batches to avoid memory issues
        batch_size = 32
        all_embeddings = []
        
        for i in range(0, len(lyrics_list), batch_size):
            batch = lyrics_list[i:i+batch_size]
            batch_embeddings = model.encode(batch, show_progress_bar=True)
            all_embeddings.append(batch_embeddings)
        
        semantic_embeddings_cache = np.vstack(all_embeddings)
        
        # Save for future use
        try:
            np.save(embeddings_file, semantic_embeddings_cache)
            print(f"Saved semantic embeddings to {embeddings_file}")
        except Exception as e:
            print(f"Could not save embeddings: {e}")
        
        return semantic_embeddings_cache
        
    except Exception as e:
        print(f"Error computing semantic embeddings: {e}")
        # Fallback to random embeddings with appropriate dimensions
        np.random.seed(42)
        semantic_embeddings_cache = np.random.randn(len(data), 384)  # all-MiniLM-L6-v2 has 384 dimensions
        return semantic_embeddings_cache

def get_hybrid_similarity_chunked(liked_idx, tfidf_weight=0.5, semantic_weight=0.5, chunk_size=1000):
    """
    Compute hybrid similarity using chunked processing to manage memory.
    """
    n_songs = len(data)
    hybrid_sim_agg = np.zeros(n_songs)
    
    # Get TF-IDF similarities
    if 'embeddings' in globals() and embeddings is not None:
        liked_tfidf = embeddings[liked_idx]
        tfidf_dists = cosine_distances(liked_tfidf, embeddings)
        tfidf_sims = 1 - tfidf_dists
        tfidf_sim_agg = np.median(tfidf_sims, axis=0)
    else:
        tfidf_sim_agg = np.zeros(n_songs)
    
    # Get semantic similarities
    try:
        semantic_embs = ensure_semantic_embeddings()
        liked_semantic = semantic_embs[liked_idx]
        semantic_dists = cosine_distances(liked_semantic, semantic_embs)
        semantic_sims = 1 - semantic_dists
        semantic_sim_agg = np.median(semantic_sims, axis=0)
    except Exception as e:
        print(f"Error computing semantic similarities: {e}")
        semantic_sim_agg = np.zeros(n_songs)
    
    # Combine TF-IDF and semantic similarities
    hybrid_sim_agg = tfidf_weight * tfidf_sim_agg + semantic_weight * semantic_sim_agg
    
    return hybrid_sim_agg

def get_hybrid_embeddings_sample(sample_indices, tfidf_weight=0.5, semantic_weight=0.5):
    """
    Create hybrid embeddings for a sample of songs by combining TF-IDF and semantic embeddings.
    """
    # Get TF-IDF embeddings for sample
    if 'embeddings' in globals() and embeddings is not None:
        tfidf_sample = embeddings[sample_indices]
        if hasattr(tfidf_sample, 'toarray'):
            tfidf_sample = tfidf_sample.toarray()
    else:
        # Fallback to zeros if TF-IDF not available
        tfidf_sample = np.zeros((len(sample_indices), 1000))  # Assume 1000 TF-IDF features
    
    # Get semantic embeddings for sample
    try:
        semantic_embs = ensure_semantic_embeddings()
        semantic_sample = semantic_embs[sample_indices]
    except Exception as e:
        print(f"Error getting semantic embeddings: {e}")
        semantic_sample = np.zeros((len(sample_indices), 384))  # Default semantic embedding size
    
    # Normalize both embeddings to unit vectors
    from sklearn.preprocessing import normalize
    tfidf_normalized = normalize(tfidf_sample, norm='l2', axis=1)
    semantic_normalized = normalize(semantic_sample, norm='l2', axis=1)
    
    # Combine embeddings with weights
    # We need to handle dimension mismatch - pad the smaller one with zeros
    tfidf_dim = tfidf_normalized.shape[1]
    semantic_dim = semantic_normalized.shape[1]
    
    if tfidf_dim > semantic_dim:
        # Pad semantic embeddings
        padding = np.zeros((len(sample_indices), tfidf_dim - semantic_dim))
        semantic_padded = np.hstack([semantic_normalized, padding])
        hybrid_embeddings = tfidf_weight * tfidf_normalized + semantic_weight * semantic_padded
    elif semantic_dim > tfidf_dim:
        # Pad TF-IDF embeddings
        padding = np.zeros((len(sample_indices), semantic_dim - tfidf_dim))
        tfidf_padded = np.hstack([tfidf_normalized, padding])
        hybrid_embeddings = tfidf_weight * tfidf_padded + semantic_weight * semantic_normalized
    else:
        # Same dimensions
        hybrid_embeddings = tfidf_weight * tfidf_normalized + semantic_weight * semantic_normalized
    
    return hybrid_embeddings

In [6]:
def get_ranked_recommendations(song_list: pd.DataFrame, n_recs: int = 10, model_type="semantic", 
                             semantic_weight: float = 0.7, hybrid_tfidf_weight: float = 0.5) -> pd.DataFrame:
    if song_list.empty:
        return pd.DataFrame(columns=['Title', 'Primary Artists', 'Featuring Artists', 'Feature_Similarity', 'Semantic_Similarity', 'Combined_Score'])

    # Special case for random: just pick random songs with no duplicates
    if model_type == 'random':
        np.random.seed(42)  # For reproducibility
        
        # Get indices of liked songs to exclude them
        liked_idx = song_list.index.to_numpy()
        
        # Exclude liked songs from potential recommendations
        available_mask = ~data.index.isin(liked_idx)
        
        # Exclude items with the same Title + Primary Artists as any liked song
        liked_pairs = set(
            zip(
                song_list['Title'].fillna('').astype(str).str.lower().str.strip(),
                song_list['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
            )
        )
        title_artist_series = pd.Series(
            list(zip(
                data['Title'].fillna('').astype(str).str.lower().str.strip(),
                data['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
            )),
            index=data.index
        )
        available_mask &= ~title_artist_series.isin(liked_pairs)
        
        available_indices = data.index[available_mask].tolist()
        
        # Randomly select songs while avoiding duplicates
        selected_idxs = []
        seen_titles_artists = set()
        
        # Shuffle available indices for random selection
        shuffled_indices = available_indices.copy()
        np.random.shuffle(shuffled_indices)
        
        for idx in shuffled_indices:
            if len(selected_idxs) >= n_recs:
                break
                
            # Get normalized title and primary artist for duplicate detection
            title = data.loc[idx, 'Title']
            primary_artist = data.loc[idx, 'Primary Artists']
            
            # Normalize for comparison (lowercase, strip whitespace, handle NaN)
            normalized_title = str(title).lower().strip() if pd.notna(title) else ''
            normalized_artist = str(primary_artist).lower().strip() if pd.notna(primary_artist) else ''
            
            title_artist_key = (normalized_title, normalized_artist)
            
            # Skip if we've already seen this title-artist combination
            if title_artist_key in seen_titles_artists:
                continue
                
            selected_idxs.append(idx)
            seen_titles_artists.add(title_artist_key)
        
        # Create recommendations DataFrame with random scores
        if selected_idxs:
            recs = data.loc[selected_idxs, ['Rank', 'Title', 'Primary Artists', 'Featuring Artists'] + features].copy()
            # Add dummy similarity scores for consistency with other models
            recs['Feature Similarity'] = np.random.rand(len(selected_idxs))
            recs['Lyric Similarity'] = np.random.rand(len(selected_idxs))
            recs['Combined Score'] = np.random.rand(len(selected_idxs))
        else:
            # Return empty DataFrame if no recommendations found
            recs = pd.DataFrame(columns=['Rank', 'Title', 'Primary Artists', 'Featuring Artists'] + features + 
                              ['Feature Similarity', 'Lyric Similarity', 'Combined Score'])
        
        recs = recs.reset_index().drop(columns=['index'])
        return recs

    # Impute & scale audio features on full dataset
    impute_vals = data[features].median()
    X_full_imputed = data[features].fillna(impute_vals)
    scaler = StandardScaler().fit(X_full_imputed)
    X_scaled = pd.DataFrame(scaler.transform(X_full_imputed), index=X_full_imputed.index, columns=features)

    liked_idx = song_list.index.to_numpy()
    liked_feat = X_scaled.loc[liked_idx].values
    feat_dists = cosine_distances(liked_feat, X_scaled.values)
    feat_sims_matrix = 1 - feat_dists
    feat_sim_agg = np.median(feat_sims_matrix, axis=0)

    # Word similarities - initialize with zeros as default
    semantic_sim_agg = np.zeros(len(data))
    
    # Hybrid: Use combination of TF-IDF and Semantic embeddings (memory-efficient)
    if model_type == 'hybrid':
        semantic_sim_agg = get_hybrid_similarity_chunked(
            liked_idx, 
            tfidf_weight=hybrid_tfidf_weight, 
            semantic_weight=1.0 - hybrid_tfidf_weight
        )
    
    # Semantic: Use semantic word-embeddings for lyrical similarity
    elif model_type == 'semantic':
        try:
            emb_dense = ensure_semantic_embeddings()
            liked_emb = emb_dense[liked_idx]
            sem_dists = cosine_distances(liked_emb, emb_dense)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        except Exception as e:
            print(f"Semantic embeddings unavailable; using zeros. Error: {str(e)}")
            semantic_sim_agg = np.zeros(len(data))

    # TF-IDF: Use word frequencies for lyrical similarity
    elif model_type == "tf-idf":
        if 'embeddings' in globals() and embeddings is not None:
            liked_emb = embeddings[liked_idx]
            sem_dists = cosine_distances(liked_emb, embeddings)
            sem_sims_matrix = 1 - sem_dists
            semantic_sim_agg = np.median(sem_sims_matrix, axis=0)
        else:
            semantic_sim_agg = np.zeros(len(data))

    # Features: Use only the audio features, no lyrical similarity
    elif model_type == 'features':
        semantic_sim_agg = np.zeros(len(data))

    # For hybrid model, semantic_weight is ignored (combination already done in embeddings)
    if model_type == 'hybrid':
        combined = 0.7 * semantic_sim_agg + 0.3 * feat_sim_agg  # Default weighting
    else:
        combined = semantic_weight * semantic_sim_agg + (1.0 - semantic_weight) * feat_sim_agg

    scores = pd.DataFrame({
        'Feature Similarity': feat_sim_agg,
        'Lyric Similarity': semantic_sim_agg,
        'Combined Score': combined
    }, index=X_scaled.index)

    # Exclude liked songs
    mask = ~scores.index.isin(liked_idx)

    # Exclude any items with 100% lyric similarity
    if 'Lyric Similarity' in scores.columns:
        mask &= scores['Lyric Similarity'] < 1.0

    # Exclude items with the same Title + Primary Artists as any liked song
    liked_pairs = set(
        zip(
            song_list['Title'].fillna('').astype(str).str.lower().str.strip(),
            song_list['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
        )
    )
    title_artist_series = pd.Series(
        list(zip(
            data['Title'].fillna('').astype(str).str.lower().str.strip(),
            data['Primary Artists'].fillna('').astype(str).str.lower().str.strip()
        )),
        index=data.index
    )
    mask &= ~title_artist_series.isin(liked_pairs)

    # Sort by combined score and select top recommendations while avoiding duplicates
    filtered_scores = scores[mask].sort_values('Combined Score', ascending=False)
    
    selected_idxs = []
    seen_titles_artists = set()
    
    for idx in filtered_scores.index:
        if len(selected_idxs) >= n_recs:
            break
            
        # Get normalized title and primary artist for duplicate detection
        title = data.loc[idx, 'Title']
        primary_artist = data.loc[idx, 'Primary Artists']
        
        # Normalize for comparison (lowercase, strip whitespace, handle NaN)
        normalized_title = str(title).lower().strip() if pd.notna(title) else ''
        normalized_artist = str(primary_artist).lower().strip() if pd.notna(primary_artist) else ''
        
        title_artist_key = (normalized_title, normalized_artist)
        
        # Skip if we've already seen this title-artist combination
        if title_artist_key in seen_titles_artists:
            continue
            
        selected_idxs.append(idx)
        seen_titles_artists.add(title_artist_key)

    top = scores.loc[selected_idxs]

    recs = data.loc[top.index, ['Rank', 'Title', 'Primary Artists', 'Featuring Artists'] + features].copy()
    recs['Feature Similarity'] = top['Feature Similarity'].values
    recs['Lyric Similarity'] = top['Lyric Similarity'].values
    recs['Combined Score'] = top['Combined Score'].values

    recs = recs.reset_index().drop(columns=['index'])
    return recs

In [7]:
# Quantitative Methods to Compare TF-IDF vs Semantic Embeddings
def evaluate_model_quality(model_type='semantic', n_samples=500, semantic_weight=0, hybrid_tfidf_weight=0.5):
    # Get embeddings based on model type
    if model_type == 'semantic':
        emb_dense = ensure_semantic_embeddings()
    elif model_type == 'tfidf':
        emb = embeddings  # TF-IDF embeddings (sparse)
        emb_dense = emb.toarray() if hasattr(emb, 'toarray') else emb
    elif model_type == 'hybrid':
        # Sample first, then create hybrid embeddings for sample only
        sample_indices = random.sample(range(len(data)), min(n_samples, len(data)))
        sample_data = data.iloc[sample_indices].copy()
        emb_dense = get_hybrid_embeddings_sample(
            sample_indices,
            tfidf_weight=hybrid_tfidf_weight, 
            semantic_weight=1.0 - hybrid_tfidf_weight
        )
        # Set sample_embeddings directly since we already have the right data
        sample_embeddings = emb_dense
    elif model_type == 'features':
        # Use only audio features as embeddings
        feature_data = data[features].fillna(data[features].median())
        scaler = StandardScaler()
        emb_dense = scaler.fit_transform(feature_data)
    elif model_type == 'random':
        # Use random embeddings as baseline
        np.random.seed(42)  # For reproducibility
        # create random "audio feature" samples matching column stats, then combine with random noise
        rng = np.random.RandomState(42)
        feat_df = data[features].dropna()
        means = feat_df.mean().values
        stds = feat_df.std().replace(0, 1e-6).values
        mins = feat_df.min().values
        maxs = feat_df.max().values

        rand_feat = rng.normal(loc=means, scale=stds, size=(len(data), len(features)))
        # clip to realistic ranges observed in the dataset
        for j in range(len(features)):
            rand_feat[:, j] = np.clip(rand_feat[:, j], mins[j], maxs[j])

        # standardize so feature columns have comparable scale with embedding noise
        scaler = StandardScaler()
        rand_feat_scaled = scaler.fit_transform(rand_feat)

        # remaining dimensions filled with Gaussian noise so total dim = 100
        remaining = max(0, 100 - rand_feat_scaled.shape[1])
        noise = rng.randn(len(data), remaining)

        # final random embeddings include randomized features + noise
        emb_dense = np.hstack([rand_feat_scaled, noise])
        emb_dense = np.random.randn(len(data), 100)  # 100-dim random vectors
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
    
    # Sample random songs for evaluation (hybrid handled separately above)
    if model_type != 'hybrid':
        sample_indices = random.sample(range(len(data)), min(n_samples, len(data)))
        sample_data = data.iloc[sample_indices].copy()
        sample_embeddings = emb_dense[sample_indices]
    
    results = {}
    
    # 1. Silhouette Score (How well separated are the clusters?)
    n_clusters = 10
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(sample_embeddings)
    
    silhouette_avg = silhouette_score(sample_embeddings, cluster_labels)
    results['silhouette_score'] = silhouette_avg
    
    # 2. Semantic Coherence Test (do similar artists cluster together?)
    artist_coherence_scores = []
    unique_artists = sample_data['Primary Artists'].value_counts()
    
    for artist, count in unique_artists.head(10).items():  # Top 10 artists
        if count >= 2:  # Artist has at least 2 songs
            artist_songs = sample_data[sample_data['Primary Artists'] == artist]
            artist_indices = [sample_indices.index(idx) for idx in artist_songs.index if idx in sample_indices]
            
            if len(artist_indices) >= 2:
                artist_embs = sample_embeddings[artist_indices]
                # Calculate average pairwise distance within artist
                distances = pairwise_distances(artist_embs, metric='cosine')
                avg_intra_distance = np.mean(distances[np.triu_indices(len(distances), k=1)])
                artist_coherence_scores.append(1 - avg_intra_distance)  # Convert to similarity
    
    avg_artist_coherence = np.mean(artist_coherence_scores) if artist_coherence_scores else 0
    results['artist_coherence'] = avg_artist_coherence
    
    # 3. Diversity (how varied are the recommendations?)
    
    # Calculate average pairwise distances
    sample_distances = pairwise_distances(sample_embeddings, metric='cosine')
    avg_distance = np.mean(sample_distances[np.triu_indices(len(sample_distances), k=1)])
    results['average_diversity'] = avg_distance
    
    # Calculate embedding dimensionality efficiency
    explained_variance = np.var(sample_embeddings, axis=0).sum()
    embedding_efficiency = silhouette_avg / (explained_variance + 1e-8)  # Add small epsilon to avoid division by zero
    results['embedding_efficiency'] = embedding_efficiency
    
    return results

In [8]:
evaluation_complete = False

In [ ]:
# Run evaluation for all models, preventing duplicate execution
if 'evaluation_complete' not in globals() or not evaluation_complete:
    n_samples = data.shape[0]
    print(f"Evaluating {n_samples} random songs from the dataset...")

    print(" - Getting TF-IDF Results...")
    tfidf_results = evaluate_model_quality('tfidf', n_samples=n_samples)
    tfidf_results_by_weight = {}
    best_tfidf_weight = None
    best_score = float('-inf')  # silhouette_score only

    # TF-IDF is slow to compute so we check fewer parameters
    # for w in sorted({0.2, 0.4, 0.6, 0.8}):
    for w in sorted({0.4}):
        print(f"    - Testing TF-IDF weight={w:.1f}...")
        res = evaluate_model_quality('tfidf', n_samples=n_samples, semantic_weight=w)
        tfidf_results_by_weight[w] = res

        sil = res.get('silhouette_score', float('-inf'))
        if sil > best_score:
            best_score = sil
            best_tfidf_weight = w

    # Use best hybrid result
    tfidf_results = tfidf_results_by_weight[best_tfidf_weight]
    print(f" - Best TF-IDF weight: {best_tfidf_weight:.1f} -> silhouette={best_score:.4f}")

    # Evaluate semantic score across weights and pick best based on silhouette_score
    print(" - Getting Semantic Results...")
    semantic_results_by_weight = {}
    best_semantic_weight = None
    best_score = float('-inf')  # silhouette_score only

    for w in sorted({0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1}):
        print(f"    - Testing semantic weight={w:.1f}...")
        res = evaluate_model_quality('semantic', n_samples=n_samples, semantic_weight=w)
        semantic_results_by_weight[w] = res

        sil = res.get('silhouette_score', float('-inf'))
        if sil > best_score:
            best_score = sil
            best_semantic_weight = w

    # Use best result
    semantic_results = semantic_results_by_weight[best_semantic_weight]
    print(f" - Best Semantic weight: {best_semantic_weight:.1f} -> silhouette={best_score:.4f}")
    
    print(" - Getting Hybrid Results")
    # Evaluate hybrid across weights and pick best based on silhouette_score
    hybrid_results_by_weight = {}
    best_hybrid_weight = None
    best_score = float('-inf')  # silhouette_score only

    for w in sorted({0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9}):
        print(f"    - Testing hybrid weight={w:.1f}...")
        res = evaluate_model_quality('hybrid', n_samples=n_samples, hybrid_tfidf_weight=w)
        hybrid_results_by_weight[w] = res

        sil = res.get('silhouette_score', float('-inf'))
        if sil > best_score:
            best_score = sil
            best_hybrid_weight = w

    # Use best hybrid result
    hybrid_results = hybrid_results_by_weight[best_hybrid_weight]
    print(f" - Best Hybrid weight: {best_hybrid_weight:.1f} -> silhouette={best_score:.4f}")

        
    print(" - Getting Features-Only Results...")
    features_results = evaluate_model_quality('features', n_samples=n_samples)
    print(" - Getting Random Baseline Results...")
    random_results = evaluate_model_quality('random', n_samples=n_samples)
    
    evaluation_complete = True
else:
    print("Evaluation already complete. Using cached results.")

Evaluating 8034 random songs from the dataset...
 - Getting TF-IDF Results...
    - Testing TF-IDF weight=0.4...
    - Testing TF-IDF weight=0.4...
 - Best TF-IDF weight: 0.4 -> silhouette=0.0158
 - Getting Semantic Results...
    - Testing semantic weight=0.0...
Loading pre-computed semantic embeddings...
 - Best TF-IDF weight: 0.4 -> silhouette=0.0158
 - Getting Semantic Results...
    - Testing semantic weight=0.0...
Loading pre-computed semantic embeddings...
    - Testing semantic weight=0.1...
    - Testing semantic weight=0.1...
    - Testing semantic weight=0.2...
    - Testing semantic weight=0.2...
    - Testing semantic weight=0.3...
    - Testing semantic weight=0.3...
    - Testing semantic weight=0.4...
    - Testing semantic weight=0.4...
    - Testing semantic weight=0.5...
    - Testing semantic weight=0.5...
    - Testing semantic weight=0.6...
    - Testing semantic weight=0.6...
    - Testing semantic weight=0.7...
    - Testing semantic weight=0.7...
    - Testing 

In [ ]:
# Mapping for display names and formatting rules (reuse existing `metrics` if present)
metrics = globals().get('metrics', [
    ('silhouette_score', 'Silhouette Score', '{:.4f}'),
    ('artist_coherence', 'Artist Coherence Score', '{:.4f}'),
    ('average_diversity', 'Average Pairwise Distance', '{:.4f}'),
    ('embedding_efficiency', 'Embedding Efficiency (Sil/Var)', '{:.4f}')
])

GREEN = '\033[92m'
WHITE = '\033[97m'
GREY = '\033[90m'
BOLD = '\033[1m'
RESET = '\033[0m'

# Display results table
if 'tfidf_results' in globals():
    header = f"{'\nMetric':40s} {'TF-IDF':12s} {'Semantic':12s} {'Hybrid':12s} {'Features':12s} {'Random':12s} {'Best'}"
    print(header)
    print("-" * len(header))

for key, display_name, _ in metrics:
    tfidf_val = tfidf_results.get(key, 0)
    semantic_val = semantic_results.get(key, 0)
    hybrid_val = hybrid_results.get(key, 0)
    features_val = features_results.get(key, 0)
    random_val = random_results.get(key, 0)
    
    # Format all values
    fmt = next(fmt for k, _, fmt in metrics if k == key)
    tfidf_str = fmt.format(tfidf_val)
    semantic_str = fmt.format(semantic_val)
    hybrid_str = fmt.format(hybrid_val)
    features_str = fmt.format(features_val)
    random_str = fmt.format(random_val)
    
    # Find the best value
    values = [tfidf_val, semantic_val, hybrid_val, features_val, random_val]
    names = ['TF-IDF', 'Semantic', 'Hybrid', 'Features', 'Random']
    if key == 'average_diversity':
        best_idx = values.index(min(values))
    else:
        best_idx = values.index(max(values))
    best_name = names[best_idx]
    
    # Color by rank: best=GREEN+BOLD, 2nd=DIM_GREEN, else=GREY
    raw_displays = [tfidf_str, semantic_str, hybrid_str, features_str, random_str]
    # Determine ranking (higher is better except for average_diversity where lower is better)
    reverse = (key != 'average_diversity')
    sorted_indices = sorted(range(len(values)), key=lambda i: values[i], reverse=reverse)
    
    displays = []
    for i, val in enumerate(raw_displays):
        # find rank of this column (0 = best)
        rank = sorted_indices.index(i)
        if rank == 0:
            color = GREEN
            prefix = BOLD
        elif rank == 1:
            color = WHITE
            prefix = ''
        else:
            color = GREY
            prefix = ''
        displays.append(f"{prefix}{color}{val:12s}{RESET}")
    
    print(f"{display_name:40s} {" ".join(displays)} {best_name}")

# Model Types:
#  - TF-IDF: Term Frequency-Inverse Document Frequency on lyrics
#  - Semantic: Sentence-transformer embeddings (all-MiniLM-L6-v2) on lyrics  
#  - Hybrid: Blend of TF-IDF and Semantic scores for Lyrics
#  - Features: Audio features only (Danceability, Energy, etc.)
#  - Random: Random embeddings (baseline control)
#
# Metrics:
#  - Silhouette Score: Measures how well-separated the clusters are in embedding space
#  - Artist Coherence: Measures whether songs by the same artist cluster together
#  - Average Pairwise Distance: Measures how spread out/diverse songs are in the embedding space
#  - Embedding Efficiency: silhouette_score/variance, high variance may be noise if clustering is poor (as in random)


In [ ]:
artist = "lana del rey"
artist_songs = get_songs_by_artist(artist)
liked_songs = artist_songs.head(1) if len(artist_songs) > 1 else artist_songs

ranked_recs_tfidf = get_ranked_recommendations(liked_songs, n_recs=10, model_type="tf-idf", semantic_weight=best_tfidf_weight)
ranked_recs_semantic = get_ranked_recommendations(liked_songs, n_recs=10, model_type="semantic", semantic_weight=best_semantic_weight)
ranked_recs_hybrid = get_ranked_recommendations(liked_songs, n_recs=10, model_type="hybrid", hybrid_tfidf_weight=best_hybrid_weight)
ranked_recs_features = get_ranked_recommendations(liked_songs, n_recs=10, model_type="features")
ranked_recs_random = get_ranked_recommendations(liked_songs, n_recs=10, model_type="random")

print(f"Artist: {artist}")
display(liked_songs["Title"])

print(f"Semantic + Feature ({round(best_semantic_weight * 100)}:{round((1 - best_semantic_weight) * 100)}) Recommendations:")
display(ranked_recs_semantic)

print(f"TF-IDF + Feature ({round(best_tfidf_weight * 100)}:{round((1 - best_tfidf_weight) * 100)}) Recommendations:")
display(ranked_recs_tfidf)

print(f"Hybrid Semantic/TF-IDF ({round(best_hybrid_weight * 100)}:{round((1 - best_hybrid_weight) * 100)}) + Feature Recommendations")
display(ranked_recs_hybrid)

print("Feature Only Recommendations")
display(ranked_recs_features)

print("Random Recommendations")
display(ranked_recs_random)

print()